# Этап 2.3. Подготовка данных

По методичке (п.4 и п.5):
1. предварительный анализ: объём, пропуски, смысл атрибутов, шкалы, роли, примеры записей;
2. очистка;
3. разметка (аннотирование) и векторизация текстов;
4. слияние;
5. хранение, скрипт слияния и обновления.

Код загрузки, очистки и слияния лежит в `etl.py`, ноутбук вызывает его функции. Период — 2019–2025 гг.

In [1]:
import os
import sqlite3
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import etl

START, END = '2019-01-01', '2025-12-31'

## 1. Загрузка

In [2]:
price = etl.load_price(START, END)          # JSON, Energy-Charts
power = etl.load_power(START, END)          # JSON, Energy-Charts
weather = etl.load_weather(START, END)      # JSON, Open-Meteo
holidays = etl.load_holidays(START, END)    # JSON, Nager.Date
eurostat = etl.load_eurostat(START, END)    # XML, Eurostat
opsd = etl.load_opsd()                      # SQLite, OPSD
news_raw = etl.load_news(START, END)        # текст, Google News

## 2. Предварительный анализ

### 2.1. Объём и пропуски

In [3]:
sources = {'цена': price, 'генерация': power, 'погода': weather, 'праздники': holidays,
           'Eurostat': eurostat, 'OPSD': opsd, 'новости': news_raw}
for name, data in sources.items():
    print(f'{name:10} {len(data):>7} строк, пропусков: {data.isna().sum().sum()}')

цена         67995 строк, пропусков: 0
генерация   245472 строк, пропусков: 214472
погода      306960 строк, пропусков: 0
праздники      135 строк, пропусков: 0
Eurostat        84 строк, пропусков: 0
OPSD         17540 строк, пропусков: 0
новости       1948 строк, пропусков: 0


In [4]:
# столбцы, где есть пропуски
power.isna().sum()[power.isna().sum() > 0]

Nuclear                     95140
Fossil coal-derived gas    119332
dtype: int64

Пропуски есть только в двух рядах генерации:
- `Nuclear` — АЭС Германии закрыты 15.04.2023, после этого ряд не публикуется. Пропуск означает ноль.
- `Fossil coal-derived gas` — ряд публикуется не за весь период и для анализа не нужен.

### 2.2. Примеры записей и распределения

In [5]:
price.describe().round(1)

count    67995.0
mean        94.6
std         90.3
min       -500.0
25%         39.4
50%         78.2
75%        113.7
max        936.3
Name: price, dtype: float64

In [6]:
power[['Load', 'Residual load', 'Solar', 'Wind onshore', 'Fossil gas']].describe().round()

,Load,Residual load,Solar,Wind onshore,Fossil gas
count,245472.0,245472.0,245472.0,245472.0,245472.0
mean,54741.0,34097.0,6037.0,11749.0,5542.0
std,9691.0,13391.0,9376.0,9313.0,3060.0
min,27933.0,-6652.0,0.0,42.0,831.0
25%,46857.0,25150.0,0.0,4439.0,3145.0
50%,54609.0,34604.0,109.0,9046.0,4968.0
75%,62269.0,43347.0,9622.0,16897.0,7233.0
max,81614.0,75212.0,50385.0,47901.0,17719.0


In [7]:
weather.head(3)

,time,temperature_2m,wind_speed_100m,shortwave_radiation,cloud_cover,weather_code,city
0,2018-12-31 00:00:00+00:00,2.9,25.4,0.0,24,1,Berlin
1,2018-12-31 01:00:00+00:00,2.8,22.1,0.0,28,1,Berlin
2,2018-12-31 02:00:00+00:00,2.0,21.6,0.0,89,3,Berlin


In [8]:
news_raw.head(3)

,title,source,pubDate,link
0,Strompreise und EE-Anlagen in der Energiewirts...,PwC,"Mon, 28 Jan 2019 16:13:14 GMT",https://news.google.com/rss/articles/CBMigwFBV...
1,Steigende Strompreise: So sparen Eigentümer - ...,Presseportal,"Thu, 31 Jan 2019 08:00:00 GMT",https://news.google.com/rss/articles/CBMiWEFVX...
2,Smogalarm in Polen - Veraltete Kohleöfen qualm...,SRF,"Tue, 22 Jan 2019 08:00:00 GMT",https://news.google.com/rss/articles/CBMioAFBV...


### 2.3. Атрибуты: смысл, шкала, роль, значимость

Гипотеза: температура, ветер, время суток и день недели позволяют предсказывать «дешёвые окна» точнее простых правил.

Шкалы — по типологии из 8 аналитических шкал:
- **описательная ветвь:** номинальная (ID), контентная;
- **смысловая ветвь:** категориальная, дихотомическая, порядковая;
- **измерительная ветвь:** интервальная, отношений, абсолютная.

| Атрибут | Смысл | Ед. | Шкала | Роль | Значимость для гипотезы |
|:--|:--|:--|:--|:--|:--|
| `link` | ссылка на новость | — | номинальная (ID) | ключ | — (уникальный номер статьи) |
| `title` | заголовок новости | — | контентная | источник признаков | низкая — из него получаются тема и сигнал |
| `time` | час наблюдения (UTC) | дата-время | интервальная | ключ слияния | высокая — общий атрибут почасовых источников |
| `date` | дата по местному времени | — | интервальная | ключ слияния | средняя — по ней присоединяются праздники и новости |
| `month` | месяц по местному времени | — | интервальная | ключ слияния | низкая — по нему присоединяются данные Eurostat |
| `price` | цена рынка на сутки вперёд | EUR/МВт·ч | отношений | целевая | высокая — основной показатель гипотезы |
| `load` | потребление | МВт | отношений | объект | высокая — спрос |
| `residual_load` | потребление минус ветер и солнце | МВт | отношений | объект | высокая — что покрывают газ и уголь |
| `res_share` | доля ВИЭ в покрытии потребления | % | отношений | фактор | высокая |
| `solar`, `wind_onshore`, `wind_offshore` | выработка солнечных и ветровых станций | МВт | отношений | фактор | высокая — дешёвая генерация |
| `gas`, `lignite`, `hard_coal` | выработка газовых и угольных станций | МВт | отношений | фактор | средняя — дорогая генерация задаёт цену |
| `nuclear`, `biomass`, `hydro` | прочая генерация | МВт | отношений | фактор | низкая — почти постоянна |
| `net_import` | сальдо перетоков с соседями (знак = направление) | МВт | отношений | фактор | средняя |
| `temperature_2m` | температура воздуха | °C | интервальная | фактор | высокая (в гипотезе) — отопление |
| `wind_speed_100m` | ветер на высоте ветрогенератора | км/ч | отношений | фактор | высокая (в гипотезе) |
| `shortwave_radiation` | солнечная радиация | Вт/м² | отношений | фактор | высокая — выработка СЭС |
| `cloud_cover` | облачность | % | отношений | фактор | низкая |
| `weather_code` | тип погоды по WMO | — | категориальная | фактор | низкая |
| `city` | город замера погоды | — | категориальная | разрез | — (до слияния) |
| `is_holiday` | федеральный праздник | да/нет | дихотомическая | фактор | средняя — спрос как в выходной |
| `eu_res_share` | доля ВИЭ в производстве за месяц | доля | отношений | фактор | низкая — сезонный фон |
| `topic` | тема новости | — | категориальная | фактор | низкая |
| `price_signal` | снижение / нейтрально / рост цен в заголовке | −1, 0, 1 | порядковая | фактор | низкая |
| `news_count` | число новостей за день | шт. | абсолютная | фактор | низкая — внешний фон |
| `news_signal` | средний ценовой сигнал новостей за день | −1…1 | отношений | фактор | низкая |

## 3. Очистка

| Проблема | Решение |
|:--|:--|
| с 01.10.2025 цена публикуется каждые 15 минут, генерация — всегда каждые 15 минут | приводим к часу средним значением |
| летнее и зимнее время: сутки из 23 и 25 часов | храним время в UTC, местное время — только для календаря |
| пропуски `Nuclear` после закрытия АЭС (15.04.2023); в выгрузке целиком после этой даты ряда нет совсем | отсутствующий ряд создаётся пустым, пропуски заполняются нулём — выработка АЭС действительно 0 МВт |
| 22 ряда генерации, часть не нужна | оставляем 13 рядов с короткими именами |
| погода в 5 городах, а цена одна на всю зону | среднее по городам; код погоды — максимальный (самое выраженное явление) |
| экстремальные цены −500 и 936 EUR/МВт·ч | оставляем: это реальные события в пределах биржевых лимитов (−500…4000) |

In [9]:
price_h = etl.clean_price(price)
power_h = etl.clean_power(power)
weather_h = etl.clean_weather(weather)

print('часов:', len(price_h))
print('пропусков после очистки:', price_h.isna().sum() + power_h.isna().sum().sum() + weather_h.isna().sum().sum())
power_h.head(3)

часов: 61368
пропусков после очистки: 0


,load,residual_load,res_share,solar,wind_onshore,wind_offshore,gas,lignite,hard_coal,nuclear,biomass,hydro,net_import
2018-12-31 23:00:00+00:00,43273.500,19940.175,70.875,0.0,20199.95,3133.400,3372.825,6407.750,2920.425,8994.100,4787.900,1824.225,-9677.6
2019-01-01 00:00:00+00:00,41631.475,16592.100,77.750,0.0,22171.70,2867.675,3057.475,5441.075,2497.025,8528.675,4785.775,1834.300,-10022.0
2019-01-01 01:00:00+00:00,40084.725,14587.075,81.600,0.0,23037.70,2459.925,2803.100,5402.275,2314.450,7947.500,4742.325,1753.100,-11210.9


Проверка физических границ — значения вне допустимых диапазонов были бы ошибками:

In [10]:
# биржевые лимиты цены — от −500 до 4000 EUR/МВт·ч
print('цена вне −500…4000:', ((price_h < -500) | (price_h > 4000)).sum())
print('потребление ≤ 0:', (power_h['load'] <= 0).sum())
print('температура вне −40…45 °C:', ((weather_h['temperature_2m'] < -40) | (weather_h['temperature_2m'] > 45)).sum())
print('отрицательные ветер или радиация:', ((weather_h['wind_speed_100m'] < 0) | (weather_h['shortwave_radiation'] < 0)).sum())

цена вне −500…4000: 0
потребление ≤ 0: 0
температура вне −40…45 °C: 0
отрицательные ветер или радиация: 0


Сверка источников: OPSD (SQLite) и Energy-Charts (JSON) пересекаются в 10.2018 – 09.2020.

In [11]:
# цена и потребление из двух источников за одни и те же часы
check = opsd.join(price_h.rename('price_ec')).join(power_h['load'].rename('load_ec')).dropna()
print('цена: корреляция', round(check['price'].corr(check['price_ec']), 3),
      '| среднее расхождение', round((check['price'] - check['price_ec']).abs().mean(), 2), 'EUR/МВт·ч')
print('потребление: корреляция', round(check['load'].corr(check['load_ec']), 3),
      '| среднее расхождение', round((check['load'] - check['load_ec']).abs().mean()), 'МВт')

цена: корреляция 1.0 | среднее расхождение 0.0 EUR/МВт·ч
потребление: корреляция 0.995 | среднее расхождение 1097 МВт


Цена совпадает полностью: оба источника берут её из ENTSO-E. Потребление расходится примерно на 2%, потому что источники по-разному считают нагрузку. В работе используется один источник — Energy-Charts.

## 4. Текст: разметка и векторизация

**Очистка:** убираем название издания из заголовка, дубликаты и заголовки не про электроэнергию, энергию, газ или цены.

**Разметка** — по спискам слов в `etl.py`:
- `topic` — тема: газ, ВИЭ и отрицательные цены, биржа, регулирование, тарифы для потребителей, прочее;
- `price_signal` — +1, если в заголовке больше слов о росте цен, −1 — о снижении, 0 — нейтрально.

In [12]:
news = etl.annotate_news(news_raw)
print(len(news_raw), '→', len(news), 'заголовков после очистки')
news[['title', 'topic', 'price_signal']].sample(8, random_state=2)

1948 → 1456 заголовков после очистки


,title,topic,price_signal
809,In Appenzell sinken die Strompreise leicht,прочее,-1
975,Strompreis: Wie Dunkelflauten den Handel von Ö...,ВИЭ и отрицательные цены,0
224,Energiepreise : Die heimlichen Energiefresser,прочее,0
1390,Winterpreise an der Strombörse – kein Tag unte...,биржа,0
588,Strompreis: EU-Länder wollen Verbraucher vor S...,тарифы для потребителей,0
120,Stromanbieter im Vergleich – Ökostrom überrasc...,ВИЭ и отрицательные цены,-1
1360,"Strompreise sinken, deine Rechnung nicht: Dies...",тарифы для потребителей,-1
1155,Netzentgelte: Bundesregierung will ab 2026 Str...,регулирование,0


**Векторизация** — TF-IDF: каждый заголовок превращается в вектор весов слов. Вес высокий, если слово часто встречается в этом заголовке и редко в остальных.

In [13]:
# служебные немецкие слова — не учитываем
stop_words = ['der', 'die', 'das', 'und', 'in', 'für', 'mit', 'auf', 'zu', 'von', 'im', 'den', 'des', 'ist',
              'nicht', 'ein', 'eine', 'sich', 'bei', 'an', 'aus', 'wie', 'was', 'so', 'es', 'um', 'am', 'dem', 'wird']
tfidf = TfidfVectorizer(stop_words=stop_words, min_df=5)   # слово должно встретиться хотя бы в 5 заголовках
vectors = tfidf.fit_transform(news['title'])
print('матрица:', vectors.shape, '— заголовков × слов')

# слова с наибольшим суммарным весом
word_weights = pd.Series(vectors.sum(axis=0).A1, index=tfidf.get_feature_names_out())
word_weights.sort_values(ascending=False).head(15).round(1)

матрица: (1456, 361) — заголовков × слов


strompreise    114.1
strompreis     108.8
strom           75.6
deutschland     36.2
2026            34.0
steigen         30.7
energie         30.4
gas             29.5
ab              26.0
warum           25.7
sinken          24.1
stadtwerke      22.7
sind            22.7
senken          22.4
gaspreise       21.2
dtype: float64

## 5. Слияние

У источников разная детализация. Общий атрибут — время; вот как каждый источник переносится на часовую сетку:

| Источник | Формат | Детализация | Общий атрибут | Как переносится на час |
|:--|:--|:--|:--|:--|
| Energy-Charts: цена, генерация | JSON | 15 мин / час | время UTC | среднее за час |
| Open-Meteo | JSON | час × город | время UTC | среднее по городам |
| Nager.Date | JSON | день | дата | значение дня — на все его часы |
| Google News | текст | статья | дата | число статей и средний сигнал за день — на все часы дня |
| Eurostat | XML | месяц | месяц | значение месяца — на все его часы |
| OPSD | SQLite | час | время UTC | только для сверки |

In [14]:
dataset = etl.build_dataset(price_h, power_h, weather_h, holidays, eurostat, news)
print(dataset.shape, '| пропусков:', dataset.isna().sum().sum())
dataset.head(3)

(61368, 26) | пропусков: 0


,time,price,load,residual_load,res_share,solar,wind_onshore,wind_offshore,gas,lignite,...,wind_speed_100m,shortwave_radiation,cloud_cover,weather_code,date,month,is_holiday,eu_res_share,news_count,news_signal
0,2018-12-31 23:00:00+00:00,28.32,43273.500,19940.175,70.875,0.0,20199.95,3133.400,3372.825,6407.750,...,23.90,0.0,96.2,3,2019-01-01,2019-01,True,0.313582,1.0,1.0
1,2019-01-01 00:00:00+00:00,10.07,41631.475,16592.100,77.750,0.0,22171.70,2867.675,3057.475,5441.075,...,25.78,0.0,97.0,51,2019-01-01,2019-01,True,0.313582,1.0,1.0
2,2019-01-01 01:00:00+00:00,-4.08,40084.725,14587.075,81.600,0.0,23037.70,2459.925,2803.100,5402.275,...,27.68,0.0,98.8,51,2019-01-01,2019-01,True,0.313582,1.0,1.0


## 6. Хранение

Итоговый набор — около 61 тыс. часов × 26 столбцов, несколько десятков МБ.

| Вариант | Плюсы | Минусы |
|:--|:--|:--|
| CSV | просто | нет типов, индексов и обновления отдельных строк |
| Parquet | быстрое чтение столбцов | обновить часть строк нельзя — только перезаписать файл |
| PostgreSQL | промышленная СУБД, используется в EcoGrid | нужен отдельный сервер |
| **SQLite** | база в одном файле: SQL, индексы, обновление строк; встроена в Python | один пишущий процесс одновременно |

**Выбор — SQLite** (`data/energy.db`): объём небольшой, сервер не нужен, новые периоды дописываются, а уточнённые перезаписываются. При необходимости таблицы переносятся в PostgreSQL без изменений.

Таблицы: `hourly` — объединённый почасовой набор, `news` — размеченные новости.

In [15]:
if os.path.exists(etl.DB):
    os.remove(etl.DB)                     # строим базу заново
etl.save(dataset, 'hourly', START, END)
etl.save(news, 'news', START, END)

In [16]:
# проверка: сколько часов и за какой период записано
con = sqlite3.connect(etl.DB)
pd.read_sql('SELECT COUNT(*) AS hours, MIN(date) AS first_day, MAX(date) AS last_day FROM hourly', con)

,hours,first_day,last_day
0,61368,2019-01-01,2025-12-31


### Скрипт обновления

`etl.py` запускается из терминала с периодом: загружает все источники, чистит, объединяет и записывает в базу. Строки за этот период сначала удаляются, поэтому повторный запуск не создаёт дублей:

```bash
python etl.py 2026-01-01 2026-01-31
```

Проверка: дописываем январь 2026 г. два раза подряд и считаем часы по годам.

In [17]:
etl.update('2026-01-01', '2026-01-31')
etl.update('2026-01-01', '2026-01-31')   # повтор — дублей быть не должно

# в каждом году должно быть 8760 часов (8784 в високосном), в январе 2026 г. — 744
pd.read_sql('SELECT substr(date, 1, 4) AS year, COUNT(*) AS hours FROM hourly GROUP BY year', con)

,year,hours
0,2019,8760
1,2020,8784
2,2021,8760
3,2022,8760
4,2023,8760
5,2024,8784
6,2025,8760
7,2026,744


## Итог

1. **Загружено 7 источников в 4 форматах:** SQLite (OPSD), XML (Eurostat), JSON (Energy-Charts, Open-Meteo, Nager.Date) и текст (Google News).
2. **Предварительный анализ.** Пропуски есть только в двух рядах генерации, и оба объяснимы. Значений вне физических границ нет. Цена в OPSD и Energy-Charts совпадает полностью, потребление расходится на 2%. Для каждого атрибута определены смысл, шкала по типологии из 8 шкал, роль и значимость для гипотезы.
3. **Очистка.** Все ряды приведены к часу в UTC, пропуски устранены, выбраны 13 рядов генерации, погода сведена к одной зоне.
4. **Текст.** Из 1948 заголовков после очистки осталось 1456. Каждый размечен темой и ценовым сигналом, векторизован TF-IDF (1456 × 361) и сведён по дням.
5. **Слияние.** Данные с детализацией 15 минут, час, день и месяц сведены в один почасовой набор: 61 368 часов × 26 столбцов, без пропусков.
6. **Хранение.** SQLite `data/energy.db`, таблицы `hourly` и `news`. Скрипт `etl.py` дописывает данные за любой период без дублей, это проверено на январе 2026 г.